In [1]:
# Cell 0 — Bootstrap: find src/ on any machine, load config + download-log helpers. (Standard house pattern.)
import sys
from pathlib import Path

# Walk up from the notebook dir to find the repo root (the folder containing .env), then add src/ to path.
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *                                      # PROJECT_ROOT, PROCESSED_DIR, CURRENT_YEAR, BROWSER_HEADERS, ...
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests                                           # CPJ is a REST API → use requests

log = load_log()                                          # load the download-log (currency tracking)
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CURRENT_YEAR (runtime-derived in config): {CURRENT_YEAR}")

Log loaded. Rows: 44
PROJECT_ROOT: C:\Users\mjbou\governance-framework
CURRENT_YEAR (runtime-derived in config): 2026


# CPJ — Committee to Protect Journalists (Notebook 36)

**Concept 23 — Media freedom & pluralism.** Press-freedom safety signals from CPJ's public data API.

**Measures (per country):**
- **Imprisoned** (lead) — journalists currently jailed in relation to their work (live census snapshot).
- **Murdered** (3-yr window) — journalists murdered, motive-confirmed, over a rolling `CURRENT_YEAR−3 … CURRENT_YEAR` window.
- **Impunity** — unsolved (complete-impunity) murders, derived from per-case records.

**Source:** CPJ WordPress REST API (`/wp-json/cpj-datamanager/v1/`) — public, no auth. Fully automated; counts are point-in-time / rolling-window and refresh on re-run.

**Output:** `data/processed/cpj_clean.csv` — one row per country, ISO3-keyed.

In [2]:
# Cell 2 — CPJ API configuration. Public WordPress REST API (no auth) at /wp-json/cpj-datamanager/v1/.
# Three measures, three endpoint shapes (confirmed by inspecting the live site's network calls):
#   imprisoned    → current annual-census snapshot, by country (LEAD measure; most responsive to policy).
#   murdered      → killed-with-motive counts, by country, Type of Death = Murder (3-yr tail measure).
#   murdered_cases→ per-case murdered records, used to derive IMPUNITY (the aggregate has no impunity field).

CPJ_API_BASE = "https://cpj.org/wp-json/cpj-datamanager/v1"

# Rolling window derived from CURRENT_YEAR (config = datetime.today().year). NEVER hardcoded — auto-rolls yearly.
# "Last 3 years" = a 3-year lookback span ending in the current year (e.g. 2026 -> 2023,2026, which the
# live site labels "between 2023 and 2026"). The killed endpoints take year as "start,end" (an inclusive range).
KILLED_START_YEAR = CURRENT_YEAR - 3
KILLED_END_YEAR   = CURRENT_YEAR

ENDPOINTS = {
    # Imprisoned: live snapshot (currently_imprisoned=1), NOT windowed — point-in-time, most current.
    "imprisoned":     f"{CPJ_API_BASE}/counts_by_country?status=Imprisoned&currently_imprisoned=1",
    # Murdered: aggregated counts by country, motive-confirmed split included in response.
    "murdered":       f"{CPJ_API_BASE}/counts_by_country_with_motive_role?status=Killed&type_of_death=Murder&year={KILLED_START_YEAR},{KILLED_END_YEAR}",
    # Murdered per-case list: paginated; carries the per-case impunity classification for deriving impunity counts.
    "murdered_cases": f"{CPJ_API_BASE}/people_list?status=Killed&type_of_death=Murder&year={KILLED_START_YEAR},{KILLED_END_YEAR}&sortBy=fullName",
}

print(f"Killed/impunity rolling window: {KILLED_START_YEAR}–{KILLED_END_YEAR}  (derived, not hardcoded)")
print("\nEndpoints:")
for k, v in ENDPOINTS.items():
    print(f"  {k}:\n    {v}")

Killed/impunity rolling window: 2023–2026  (derived, not hardcoded)

Endpoints:
  imprisoned:
    https://cpj.org/wp-json/cpj-datamanager/v1/counts_by_country?status=Imprisoned&currently_imprisoned=1
  murdered:
    https://cpj.org/wp-json/cpj-datamanager/v1/counts_by_country_with_motive_role?status=Killed&type_of_death=Murder&year=2023,2026
  murdered_cases:
    https://cpj.org/wp-json/cpj-datamanager/v1/people_list?status=Killed&type_of_death=Murder&year=2023,2026&sortBy=fullName


In [3]:
# Cell 3 — Fetch helper: GET a CPJ API endpoint, validate, return parsed JSON. Fails LOUDLY on any problem.
# Why strict: the CPJ WP API returns HTTP 500 ("critical error") on bad params, and could return an error
# object with 200. We must not silently build on an empty/error payload — so we check status, content-type,
# the WP-error shape, and emptiness, raising a clear exception in each case.

def cpj_get(url, expect_list=True, timeout=30):
    """GET a CPJ API URL and return parsed JSON. Raises RuntimeError with a clear message on any failure."""
    resp = requests.get(url, headers=BROWSER_HEADERS, timeout=timeout)

    # 1. HTTP status must be 200. A 500 here is the WordPress "critical error" we saw on malformed params.
    if resp.status_code != 200:
        raise RuntimeError(f"CPJ API returned HTTP {resp.status_code} for:\n  {url}\n  Body (first 300 chars): {resp.text[:300]}")

    # 2. Response must be JSON, not an HTML error page.
    ctype = resp.headers.get("Content-Type", "")
    if "application/json" not in ctype:
        raise RuntimeError(f"CPJ API returned non-JSON ({ctype}) for:\n  {url}\n  Body (first 300 chars): {resp.text[:300]}")

    data = resp.json()

    # 3. WordPress REST errors come back as a dict with a 'code'/'message' shape — catch that.
    if isinstance(data, dict) and ("code" in data and "message" in data):
        raise RuntimeError(f"CPJ API returned a WP error for:\n  {url}\n  {data.get('code')}: {data.get('message')}")

    # 4. If we expect a list of country rows, an empty list means a query that matched nothing — flag it.
    if expect_list and isinstance(data, list) and len(data) == 0:
        print(f"⚠️  CPJ API returned an EMPTY list for:\n  {url}\n  (zero matching rows — verify params if unexpected)")

    return data

print("cpj_get() defined.")

cpj_get() defined.


In [4]:
# Cell 4 — Fetch IMPRISONED counts by country (lead measure: journalists currently jailed for their work).
# Endpoint returns a list of {country, z, code} where z = count, code = ISO-2. Live census snapshot.

imp_raw = cpj_get(ENDPOINTS["imprisoned"])               # list of dicts; cpj_get raises on any API failure

# Build a tidy frame. Defensive: assert the expected keys exist before trusting the structure.
assert isinstance(imp_raw, list) and len(imp_raw) > 0, "Imprisoned response is empty or not a list"
_sample_keys = set(imp_raw[0].keys())
assert {"country", "z", "code"}.issubset(_sample_keys), f"Unexpected imprisoned keys: {_sample_keys}"

imp = pd.DataFrame(imp_raw)[["country", "code", "z"]].rename(columns={"z": "cpj_imprisoned", "code": "iso2"})

# The count column should be integer; coerce and verify no nulls crept in.
imp["cpj_imprisoned"] = pd.to_numeric(imp["cpj_imprisoned"], errors="coerce").astype("Int64")
assert imp["cpj_imprisoned"].notna().all(), "Null imprisoned counts after numeric coercion"

print(f"Imprisoned: {len(imp)} countries with ≥1 jailed journalist")
print(f"Total jailed journalists (sum): {int(imp['cpj_imprisoned'].sum())}")
print(f"Null/missing ISO-2 codes: {imp['iso2'].isna().sum() + (imp['iso2'] == '').sum()}")
print("\nTop 10 by imprisoned count:")
print(imp.sort_values('cpj_imprisoned', ascending=False).head(10).to_string(index=False))

Imprisoned: 37 countries with ≥1 jailed journalist
Total jailed journalists (sum): 325
Null/missing ISO-2 codes: 1

Top 10 by imprisoned count:
                                      country iso2  cpj_imprisoned
                                        China   CN              50
Israel and the Occupied Palestinian Territory   IL              34
                                       Russia   RU              30
                                   Azerbaijan   AZ              24
                                      Belarus   BY              21
                                        Egypt   EG              18
                                      Myanmar   MM              18
                                      Vietnam   VN              16
                                      Eritrea   ER              16
                                         Iran   IR              11


In [5]:
# Cell 5 — Fetch MURDERED counts by country (3-yr rolling window, motive = Murder, confirmed work-related).
# Endpoint returns {country, z, motiveConfirmed, motiveUnconfirmed, MediaWorker, code}.
#   z               = total murdered (journalists) in window
#   motiveConfirmed = subset confirmed work-related (the cleaner signal; z may include unconfirmed)
# We keep both z (total) and motiveConfirmed so the metric pass can choose the stricter or looser count.

mur_raw = cpj_get(ENDPOINTS["murdered"])                 # list of dicts; raises on API failure

assert isinstance(mur_raw, list) and len(mur_raw) > 0, "Murdered response is empty or not a list"
_mk = set(mur_raw[0].keys())
assert {"country", "z", "code"}.issubset(_mk), f"Unexpected murdered keys: {_mk}"

mur = pd.DataFrame(mur_raw)
# Keep total + confirmed-motive subset; rename to clear column names. motiveConfirmed may be absent on some rows.
keep = {"country": "country", "code": "iso2", "z": "cpj_murdered_total"}
if "motiveConfirmed" in mur.columns:
    keep["motiveConfirmed"] = "cpj_murdered_confirmed"
mur = mur[list(keep)].rename(columns=keep)

# Coerce counts to integers; verify totals are non-null.
for c in [col for col in ["cpj_murdered_total", "cpj_murdered_confirmed"] if col in mur.columns]:
    mur[c] = pd.to_numeric(mur[c], errors="coerce").astype("Int64")
assert mur["cpj_murdered_total"].notna().all(), "Null murdered totals after coercion"

print(f"Murdered ({KILLED_START_YEAR}–{KILLED_END_YEAR}): {len(mur)} countries with ≥1 murdered journalist")
print(f"Total murdered (sum of z): {int(mur['cpj_murdered_total'].sum())}")
if "cpj_murdered_confirmed" in mur.columns:
    print(f"Total motive-confirmed (sum): {int(mur['cpj_murdered_confirmed'].sum())}")
print(f"Null/missing ISO-2 codes: {mur['iso2'].isna().sum() + (mur['iso2'] == '').sum()}")
print("\nTop 10 by murdered total:")
print(mur.sort_values('cpj_murdered_total', ascending=False).head(10).to_string(index=False))

Murdered (2023–2026): 27 countries with ≥1 murdered journalist
Total murdered (sum of z): 100
Total motive-confirmed (sum): 99
Null/missing ISO-2 codes: 0

Top 10 by murdered total:
                                      country iso2  cpj_murdered_total  cpj_murdered_confirmed
Israel and the Occupied Palestinian Territory   IL                  32                      32
                                        Yemen   YE                  20                      20
                                      Lebanon   LB                   7                       7
                                     Pakistan   PK                   4                       4
                                      Myanmar   MM                   3                       3
                                        Haiti   HT                   3                       3
                                  Philippines   PH                   3                       2
                                        India   IN        

In [9]:
# Cell 6 — Derive IMPUNITY: count of unsolved (Complete Impunity) murders per country, over the 3-yr window.
# people_list returns paginated dicts: {rowCount, pageNum, pageSize, pageCount, data:[...cases]}.
# Per-case fields used: impunity ('Complete Impunity' = unsolved), location (country NAME string).
# We loop all pages (pageCount, derived — not hardcoded), tally Complete-Impunity per location, then map
# location name -> iso2 using the murdered aggregate's country->code mapping (same CPJ names, so exact join).

import time

# --- 7a. Fetch page 1 to learn pageCount, then loop remaining pages. ---
all_cases = []
_p1 = cpj_get(ENDPOINTS["murdered_cases"] + "&page=1", expect_list=False)
total_pages = int(_p1["pageCount"])                       # derived from the API, not hardcoded
total_rows  = int(_p1["rowCount"])
all_cases.extend(_p1["data"])

for page in range(2, total_pages + 1):                    # pages 2..pageCount
    resp = cpj_get(ENDPOINTS["murdered_cases"] + f"&page={page}", expect_list=False)
    all_cases.extend(resp["data"])
    time.sleep(0.3)                                       # be polite to the API between page calls

# Integrity: we should have collected exactly rowCount cases.
assert len(all_cases) == total_rows, f"Collected {len(all_cases)} cases but API reported rowCount={total_rows}"
print(f"Collected {len(all_cases)} murdered cases across {total_pages} pages (rowCount={total_rows}).")

# --- 7b. Tally Complete-Impunity (unsolved) murders per country NAME. ---
# 'Complete Impunity' = no convictions = unsolved. Empty string = not complete impunity. Exact-match only.
imp_counts = {}
for c in all_cases:
    name = (c.get("location") or "").strip()
    if not name:
        continue
    is_unsolved = (c.get("impunity") == "Complete Impunity")
    rec = imp_counts.setdefault(name, {"cpj_murders_total_cases": 0, "cpj_murders_unsolved": 0})
    rec["cpj_murders_total_cases"] += 1                   # total murder cases (case-level; sanity vs aggregate z)
    rec["cpj_murders_unsolved"]    += int(is_unsolved)    # subset that are Complete Impunity

impunity_df = (
    pd.DataFrame.from_dict(imp_counts, orient="index")
      .reset_index().rename(columns={"index": "country"})
)

# --- 7c. Map country NAME -> iso2 using the murdered aggregate (same CPJ names; exact join, no fuzzy). ---
name_to_iso2 = dict(zip(mur["country"], mur["iso2"]))     # mur has both country name and iso2
impunity_df["iso2"] = impunity_df["country"].map(name_to_iso2)

# Flag any country name that didn't map (would indicate a name mismatch between case-list and aggregate).
unmapped = impunity_df[impunity_df["iso2"].isna() | (impunity_df["iso2"] == "")]["country"].tolist()
if unmapped:
    print(f"⚠️  {len(unmapped)} impunity country name(s) not matched to iso2 (need attention): {unmapped}")
else:
    print("All impunity country names mapped to iso2 via the murdered aggregate.")

print(f"\nCountries with ≥1 unsolved (Complete Impunity) murder: {(impunity_df['cpj_murders_unsolved'] > 0).sum()}")
print(f"Total unsolved murders (sum): {int(impunity_df['cpj_murders_unsolved'].sum())}")
print(f"Case-level total murders (should ≈ aggregate's {int(mur['cpj_murdered_total'].sum())}): {int(impunity_df['cpj_murders_total_cases'].sum())}")
print("\nTop 10 by unsolved-murder count:")
print(impunity_df.sort_values('cpj_murders_unsolved', ascending=False).head(10).to_string(index=False))

Collected 100 murdered cases across 5 pages (rowCount=100).
All impunity country names mapped to iso2 via the murdered aggregate.

Countries with ≥1 unsolved (Complete Impunity) murder: 24
Total unsolved murders (sum): 96
Case-level total murders (should ≈ aggregate's 100): 100

Top 10 by unsolved-murder count:
                                      country  cpj_murders_total_cases  cpj_murders_unsolved iso2
Israel and the Occupied Palestinian Territory                       32                    32   IL
                                        Yemen                       20                    20   YE
                                      Lebanon                        7                     7   LB
                                     Pakistan                        4                     4   PK
                                  Philippines                        3                     3   PH
                                        India                        3                     3   IN
 

In [10]:
# Cell 7 — Merge imprisoned + murdered + impunity into one per-country frame; harmonise to ISO3.
# Design decisions (flagged):
#  - Merge on COUNTRY NAME (outer), not iso2, so the UAE row (iso2=null from CPJ) is not dropped.
#  - Absence from a measure's list = TRUE ZERO (CPJ's count endpoints only return countries with >=1),
#    so we fillna(0) on the count columns — a country with jailed-but-no-murdered journalists really has 0 murdered.
#  - iso2 -> iso3 via pycountry, with a name-based override for the null iso2 (UAE) and any special cases.
import pycountry

# --- 7a. Normalise empty-string iso2 to NaN across the three frames so coalescing works. ---
for _df in (imp, mur, impunity_df):
    _df["iso2"] = _df["iso2"].replace("", pd.NA)

# --- 7b. Outer-merge the three measures on country name. iso2 columns get suffixed/kept, coalesced below. ---
m = (
    imp[["country", "iso2", "cpj_imprisoned"]]
    .merge(mur[["country", "iso2", "cpj_murdered_total", "cpj_murdered_confirmed"]],
           on="country", how="outer", suffixes=("_imp", "_mur"))
    .merge(impunity_df[["country", "iso2", "cpj_murders_total_cases", "cpj_murders_unsolved"]],
           on="country", how="outer")
)

# Coalesce the three iso2 columns (iso2_imp, iso2_mur, iso2) into one, preferring any non-null.
m["iso2"] = m["iso2_imp"].combine_first(m["iso2_mur"]).combine_first(m["iso2"])
m = m.drop(columns=["iso2_imp", "iso2_mur"])

# --- 7c. Count columns: absence from a measure = true zero. Fill and cast to integer. ---
count_cols = ["cpj_imprisoned", "cpj_murdered_total", "cpj_murdered_confirmed",
              "cpj_murders_total_cases", "cpj_murders_unsolved"]
for c in count_cols:
    m[c] = pd.to_numeric(m[c], errors="coerce").fillna(0).astype(int)

# Integrity: case-level murder total must equal the aggregate murdered total per country (same source/window).
_mismatch = m[(m["cpj_murders_total_cases"] != m["cpj_murdered_total"]) &
              (m["cpj_murders_total_cases"] > 0) & (m["cpj_murdered_total"] > 0)]
if len(_mismatch):
    print(f"⚠️  {len(_mismatch)} country(ies) where case-level murders != aggregate murders (investigate):")
    print(_mismatch[["country", "cpj_murdered_total", "cpj_murders_total_cases"]].to_string(index=False))
else:
    print("Case-level vs aggregate murder counts agree where both present.")
# total_cases was only for that cross-check; drop it (redundant with cpj_murdered_total).
m = m.drop(columns=["cpj_murders_total_cases"])

# --- 7d. iso2 -> iso3. pycountry for standard codes; name override for null/special cases. ---
NAME_TO_ISO3_OVERRIDES = {
    "United Arab Emirates": "ARE",       # CPJ returns null iso2 for UAE
    # add here if a future refresh introduces another null/non-standard iso2
}

def to_iso3(row):
    iso2 = row["iso2"]
    if isinstance(iso2, str) and iso2:
        c = pycountry.countries.get(alpha_2=iso2)
        if c:
            return c.alpha_3
    return NAME_TO_ISO3_OVERRIDES.get(row["country"])     # fallback by name; None if unresolved

m["iso3"] = m.apply(to_iso3, axis=1)

# --- 7e. Flag any unresolved iso3 — nothing dropped silently. ---
unresolved = m[m["iso3"].isna()][["country", "iso2"]]
if len(unresolved):
    print(f"\n⚠️  {len(unresolved)} country(ies) did NOT resolve to ISO3 (add an override):")
    print(unresolved.to_string(index=False))
else:
    print("All countries resolved to ISO3.")

# Duplicate iso3 check (two CPJ names collapsing to one country).
_dups = m.loc[m["iso3"].notna(), "iso3"].value_counts()
_dups = _dups[_dups > 1]
print(f"Duplicate iso3 codes: {_dups.to_dict() if len(_dups) else 'none'}")

print(f"\nMerged: {len(m)} countries appear in >=1 CPJ measure.")
print(f"  with imprisoned>0: {(m['cpj_imprisoned']>0).sum()}")
print(f"  with murdered>0:   {(m['cpj_murdered_total']>0).sum()}")
print(f"  with unsolved>0:   {(m['cpj_murders_unsolved']>0).sum()}")

Case-level vs aggregate murder counts agree where both present.
All countries resolved to ISO3.
Duplicate iso3 codes: none

Merged: 50 countries appear in >=1 CPJ measure.
  with imprisoned>0: 37
  with murdered>0:   27
  with unsolved>0:   24


In [11]:
# Cell 8 — Finalise, validate, derive as-of date, save cpj_clean.csv (overwrites existing).

# --- 8a. Final column order: keys, then the measures grouped (imprisoned | murdered | impunity). ---
ordered = [
    "iso3", "country", "iso2",
    "cpj_imprisoned",                                     # lead: jailed journalists (live snapshot)
    "cpj_murdered_total", "cpj_murdered_confirmed",       # tail: murdered, 3-yr window (total + motive-confirmed)
    "cpj_murders_unsolved",                               # impunity: Complete-Impunity murders, 3-yr (slow-moving)
]
missing = [c for c in ordered if c not in m.columns]
assert not missing, f"Expected columns missing: {missing}"
cpj_out = m[ordered].copy().sort_values("iso3").reset_index(drop=True)

# --- 8b. Integrity guards — fail loudly rather than save a broken file. ---
assert cpj_out["iso3"].notna().all(),            "Null iso3 present"
assert cpj_out["iso3"].is_unique,                "Duplicate iso3 — one-row-per-country violated"
# All measure counts must be non-negative integers.
_meas = ["cpj_imprisoned", "cpj_murdered_total", "cpj_murdered_confirmed", "cpj_murders_unsolved"]
assert (cpj_out[_meas] >= 0).all().all(),        "Negative count present"
# Logical guard: confirmed murders <= total murders; unsolved <= total murders.
assert (cpj_out["cpj_murdered_confirmed"] <= cpj_out["cpj_murdered_total"]).all(), "confirmed > total murders"
assert (cpj_out["cpj_murders_unsolved"]   <= cpj_out["cpj_murdered_total"]).all(), "unsolved > total murders"
print(f"Integrity checks passed: {len(cpj_out)} countries, {cpj_out.shape[1]} columns")

# --- 8c. Derive as-of metadata FROM runtime/data — never hardcoded. ---
# Imprisoned = live snapshot at retrieval; murdered/impunity = rolling window end = CURRENT_YEAR.
retrieval_date = datetime.today().strftime("%Y-%m-%d")
data_as_of = (f"imprisoned: live snapshot {retrieval_date}; "
              f"murdered/impunity: {KILLED_START_YEAR}-{KILLED_END_YEAR} window")
print(f"Data as-of: {data_as_of}")

# --- 8d. Save to processed, overwriting any existing copy. ---
out_path = os.path.join(PROCESSED_DIR, "cpj_clean.csv")
cpj_out.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}  ({os.path.getsize(out_path):,} bytes)")
print(f"\nFull dataset ({len(cpj_out)} countries):")
print(cpj_out.to_string(index=False))


Integrity checks passed: 50 countries, 7 columns
Data as-of: imprisoned: live snapshot 2026-06-30; murdered/impunity: 2023-2026 window

Saved → C:\Users\mjbou\governance-framework\data\processed\cpj_clean.csv  (1,392 bytes)

Full dataset (50 countries):
iso3                                       country iso2  cpj_imprisoned  cpj_murdered_total  cpj_murdered_confirmed  cpj_murders_unsolved
 AFG                                   Afghanistan   AF               1                   1                       1                     1
 ARE                          United Arab Emirates  NaN               1                   0                       0                     0
 AZE                                    Azerbaijan   AZ              24                   0                       0                     0
 BEN                                         Benin   BJ               1                   0                       0                     0
 BFA                                  Burkina Faso   BF 

In [12]:
# Cell 9 — Record this build in the download-log and source-registry (house pattern).
# CPJ is an AUTOMATED API source (no manual download): currency and counts derived from the data/runtime.

retrieval_date = datetime.today().strftime("%Y-%m-%d")
n_countries  = len(cpj_out)
n_imprisoned = int((cpj_out["cpj_imprisoned"] > 0).sum())
n_murdered   = int((cpj_out["cpj_murdered_total"] > 0).sum())

# --- 9a. Download log ---
update_entry(
    "CPJ",
    last_successful_download_date=retrieval_date,
    data_as_of_date=f"imprisoned: live snapshot {retrieval_date}; murdered/impunity: {KILLED_START_YEAR}-{KILLED_END_YEAR} window",
    local_filename="cpj_clean.csv",
    latest_available_version=f"CPJ API pull {retrieval_date} ({n_countries} countries)",
    notes=(
        "Committee to Protect Journalists (press-freedom safety, Concept 23). AUTOMATED — public WordPress "
        "REST API at /wp-json/cpj-datamanager/v1/ (no auth). Three measures per country: cpj_imprisoned "
        "(live census snapshot of jailed journalists), cpj_murdered_total/_confirmed (murdered journalists, "
        "rolling CURRENT_YEAR-3..CURRENT_YEAR window, motive=Murder), cpj_murders_unsolved (Complete-Impunity "
        "subset, same window — slow-moving, near-collinear with murdered over short window). ISO2 from API; "
        "ISO3 via pycountry + UAE override. Israel/OPT lumped under ISR by CPJ. Window auto-rolls (derived, "
        "not hardcoded). Re-run notebook to refresh."
    ),
)
print_entry("CPJ")

# --- 9b. Source registry (read-update-or-append-write; matches house pattern) ---
registry_path = os.path.join(PROCESSED_DIR, "source_registry.csv")
registry_df = pd.read_csv(registry_path)

cpj_notes = (
    "CPJ press-freedom safety data (Concept 23). AUTOMATED public REST API (/wp-json/cpj-datamanager/v1/). "
    "Measures: journalists imprisoned (live snapshot), murdered (3-yr rolling window, motive=Murder), and "
    "unsolved/Complete-Impunity murders (same window). 50 countries with >=1 incident; rest are true zeros "
    "(tail-severity signal). ISO2 from API; ISO3 via pycountry (+UAE override). Israel/OPT under ISR."
)
approach = (
    "requests GET to CPJ WP REST API; 3 endpoints (counts_by_country imprisoned; "
    "counts_by_country_with_motive_role killed/murder; paginated people_list for impunity); "
    "rolling year window from CURRENT_YEAR; iso2->iso3 via pycountry"
)

if (registry_df["source_id"] == "CPJ").any():
    registry_df.loc[registry_df["source_id"] == "CPJ", "access_method"]   = "automated_api"
    registry_df.loc[registry_df["source_id"] == "CPJ", "python_approach"] = approach
    registry_df.loc[registry_df["source_id"] == "CPJ", "notes"]           = cpj_notes
    print("Updated CPJ registry row")
else:
    registry_df = pd.concat([registry_df, pd.DataFrame([{
        "source_id": "CPJ", "access_method": "automated_api",
        "python_approach": approach, "notes": cpj_notes}])], ignore_index=True)
    print("Added CPJ registry row")

registry_df.to_csv(registry_path, index=False)
print(registry_df[registry_df["source_id"] == "CPJ"][["source_id", "access_method"]].to_string(index=False))

[download_log] Updated entry for CPJ
  source_id: CPJ
  last_attempted_date: 2026-06-30
  last_successful_download_date: 2026-06-30
  data_as_of_date: imprisoned: live snapshot 2026-06-30; murdered/impunity: 2023-2026 window
  local_filename: cpj_clean.csv
  latest_available_version: CPJ API pull 2026-06-30 (50 countries)
  no_update_reason: nan
  notes: Committee to Protect Journalists (press-freedom safety, Concept 23). AUTOMATED — public WordPress REST API at /wp-json/cpj-datamanager/v1/ (no auth). Three measures per country: cpj_imprisoned (live census snapshot of jailed journalists), cpj_murdered_total/_confirmed (murdered journalists, rolling CURRENT_YEAR-3..CURRENT_YEAR window, motive=Murder), cpj_murders_unsolved (Complete-Impunity subset, same window — slow-moving, near-collinear with murdered over short window). ISO2 from API; ISO3 via pycountry + UAE override. Israel/OPT lumped under ISR by CPJ. Window auto-rolls (derived, not hardcoded). Re-run notebook to refresh.
Updated 

In [13]:
# Cell 10 — Build summary: human-readable readout of the final cpj_clean.csv.
print("=" * 64)
print("CPJ — COMMITTEE TO PROTECT JOURNALISTS — BUILD SUMMARY")
print("=" * 64)
print(f"Output:            cpj_clean.csv  ({len(cpj_out)} countries, {cpj_out.shape[1]} columns)")
print(f"Window (killed):   {KILLED_START_YEAR}-{KILLED_END_YEAR} (rolling, derived from CURRENT_YEAR)")
print(f"Imprisoned:        live snapshot as of {datetime.today().strftime('%Y-%m-%d')}")
print()
print("Coverage (countries with >=1 in each measure):")
print(f"  imprisoned > 0:  {(cpj_out['cpj_imprisoned']>0).sum()}")
print(f"  murdered   > 0:  {(cpj_out['cpj_murdered_total']>0).sum()}")
print(f"  unsolved   > 0:  {(cpj_out['cpj_murders_unsolved']>0).sum()}")
print(f"  in >=1 measure:  {len(cpj_out)}  (all other framework countries are true zeros)")
print()
print("Global totals:")
print(f"  journalists imprisoned (sum):       {int(cpj_out['cpj_imprisoned'].sum())}")
print(f"  journalists murdered, window (sum): {int(cpj_out['cpj_murdered_total'].sum())}")
print(f"  unsolved (Complete Impunity, sum):  {int(cpj_out['cpj_murders_unsolved'].sum())}")
print()
print("Top 5 — imprisoned (detention-mode repression):")
print(cpj_out.nlargest(5, 'cpj_imprisoned')[['iso3','country','cpj_imprisoned']].to_string(index=False))
print("\nTop 5 — murdered (lethal-violence/impunity mode):")
print(cpj_out.nlargest(5, 'cpj_murdered_total')[['iso3','country','cpj_murdered_total','cpj_murders_unsolved']].to_string(index=False))
print()
print("NOTE for metric pass: imprisoned (live) and murdered/impunity (3-yr window) are on different time")
print("bases; impunity is ~collinear with murdered over this short window (slow-moving); Israel/OPT under ISR.")

CPJ — COMMITTEE TO PROTECT JOURNALISTS — BUILD SUMMARY
Output:            cpj_clean.csv  (50 countries, 7 columns)
Window (killed):   2023-2026 (rolling, derived from CURRENT_YEAR)
Imprisoned:        live snapshot as of 2026-06-30

Coverage (countries with >=1 in each measure):
  imprisoned > 0:  37
  murdered   > 0:  27
  unsolved   > 0:  24
  in >=1 measure:  50  (all other framework countries are true zeros)

Global totals:
  journalists imprisoned (sum):       325
  journalists murdered, window (sum): 100
  unsolved (Complete Impunity, sum):  96

Top 5 — imprisoned (detention-mode repression):
iso3                                       country  cpj_imprisoned
 CHN                                         China              50
 ISR Israel and the Occupied Palestinian Territory              34
 RUS                                        Russia              30
 AZE                                    Azerbaijan              24
 BLR                                       Belarus          